# Regime-Aware PPO for Portfolio Allocation

A PPO agent that allocates a long-only portfolio across a few large-cap stocks, 
with a custom Gym environment that is **aware of the market regime** (low vol / high vol / crash) 
and pays **turnover-based transaction costs**.

## Why this notebook is structured the way it is

The easy trap with RL-for-trading is to train and evaluate on the *same* price history. 
During a 2018-2023 mega-cap bull run, almost any long-only policy 'makes money' in-sample — 
the number is meaningless. So here:

- Data is split **chronologically**: train on `2018-01-01 .. 2022-01-01`, hold out `2022-01-01 .. 2024-01-01`.
- The agent is evaluated **out-of-sample** on the held-out window only.
- It is compared against two honest baselines on the same window and cost model: 
  **equal-weight (rebalanced daily)** and **buy-and-hold**.
- Regime thresholds are fixed on the training set and reused on the test set (no lookahead).

The reusable code lives in [`src/portfolio_rl/`](src/portfolio_rl); this notebook is the narrative. 
You can also run it headless: `python train.py && python evaluate.py`.


In [ ]:
# One-time install (uncomment). Pinned to match the classic gym <= 0.25 API that SB3 2.1 uses.
# !pip install -r requirements.txt


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from src.portfolio_rl import PortfolioEnv, baselines, metrics
from src.portfolio_rl.data import download_prices, train_test_split

TICKERS = ("AAPL", "MSFT", "GOOGL")
DATA_START, DATA_END = "2018-01-01", "2024-01-01"
SPLIT_DATE = "2022-01-01"   # train < split, test >= split
WINDOW, VOL_WINDOW = 30, 20
INITIAL, COST = 1_000_000, 0.001
SEED = 42
np.random.seed(SEED)


## 1. Load prices and split chronologically


In [ ]:
prices, dates = download_prices(TICKERS, DATA_START, DATA_END)
(train_prices, train_dates), (test_prices, test_dates) = train_test_split(prices, dates, SPLIT_DATE)
print(f"train: {train_dates[0].date()} .. {train_dates[-1].date()}  ({len(train_prices)} days)")
print(f"test:  {test_dates[0].date()} .. {test_dates[-1].date()}  ({len(test_prices)} days)")


## 2. Train PPO on the training window only

The regime baseline volatility computed here is saved and reused for the test set so the 
test regimes are labelled with information available *before* the test period.


In [ ]:
import torch
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
torch.manual_seed(SEED)

train_env = PortfolioEnv(train_prices, dates=train_dates, window_size=WINDOW,
                         vol_window=VOL_WINDOW, initial_capital=INITIAL,
                         transaction_cost=COST, seed=SEED)
train_baseline_vol = train_env.baseline_vol  # reuse on test to avoid lookahead

model = PPO("MlpPolicy", DummyVecEnv([lambda: train_env]), verbose=0, seed=SEED,
            learning_rate=3e-4, batch_size=64, n_steps=2048, gamma=0.99)
model.learn(total_timesteps=100_000)  # bump up for stronger policies


## 3. Evaluate out-of-sample vs baselines

The agent and both baselines run on the **held-out** test prices, same cost model, same window.


In [ ]:
def run_agent(model, prices, dates, baseline_vol):
    env = PortfolioEnv(prices, dates=dates, window_size=WINDOW, vol_window=VOL_WINDOW,
                       baseline_vol=baseline_vol, initial_capital=INITIAL,
                       transaction_cost=COST, seed=SEED)
    obs, done, curve = env.reset(), False, [env.initial_capital]
    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, _, done, info = env.step(action)
        curve.append(info["portfolio_value"])
    return np.array(curve), env.avg_turnover

agent_curve, agent_turnover = run_agent(model, test_prices, test_dates, train_baseline_vol)

test_returns = test_prices[1:] / test_prices[:-1] - 1.0
eq_curve = baselines.equal_weight(test_returns, WINDOW, INITIAL, COST)
bh_curve = baselines.buy_and_hold(test_returns, WINDOW, INITIAL, COST)

rows = {
    "PPO (regime-aware)": metrics.summary(agent_curve, turnover=agent_turnover),
    "Equal-weight":       metrics.summary(eq_curve),
    "Buy-and-hold":       metrics.summary(bh_curve),
}
for name, m in rows.items():
    print(f"{name:<20} totRet={m['total_return']*100:6.1f}%  Sharpe={m['annualized_sharpe']:5.2f}  "
          f"maxDD={m['max_drawdown']*100:6.1f}%  final=${m['final_value']:,.0f}")


## 4. Equity curves


In [ ]:
plt.figure(figsize=(11, 5))
plt.plot(agent_curve, label="PPO (regime-aware)")
plt.plot(eq_curve, label="Equal-weight", linestyle="--")
plt.plot(bh_curve, label="Buy-and-hold", linestyle=":")
plt.title("Out-of-sample equity curves (test window)")
plt.xlabel("Trading day"); plt.ylabel("Portfolio value ($)")
plt.legend(); plt.grid(True, alpha=0.3); plt.show()


## 5. Reading the result & limitations

- The row that matters is **PPO vs Buy-and-hold, out-of-sample**. Beating buy-and-hold on 
  risk-adjusted return (Sharpe) net of costs is the real bar; matching it is not a win.
- **3-ticker mega-cap universe** — survivorship-biased and highly correlated; not a claim about live trading.
- **Single test period** — one regime path. Walk-forward / multiple splits would be the next step.
- Reward is log-return net of turnover cost; no explicit risk penalty in the objective yet.

This is a research exercise in RL environment design and honest evaluation, not financial advice.
